In [1]:
import numpy as np
import tifffile
import torch
from pathlib import Path
import matplotlib.pyplot as plt
from skimage.measure import regionprops

In [2]:
RAW_DIR = Path("/QRISdata/Q6080/Q6080/IndividualLabFolders/AG/DRGAnushka/resonant")
MASK_DIR = Path.home() / "cellpose_masks_resonant"

raw_files = sorted([f for f in RAW_DIR.glob("*.tif") if not f.name.startswith(".")])
print("Raw files:", len(raw_files))

Raw files: 22


In [4]:
from segment_anything import sam_model_registry, SamPredictor

CKPT_PATH = Path.home() / "sam_checkpoints" / "sam_vit_b_01ec64.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"

sam = sam_model_registry["vit_b"](checkpoint=str(CKPT_PATH))
sam.to(device=device)
predictor = SamPredictor(sam)

print("SAM ready on", device)

SAM ready on cuda


In [5]:
raw_fp = raw_files[0]
mask_fp = MASK_DIR / f"{raw_fp.stem}_masks.tif"

raw = tifffile.imread(raw_fp)
cp_masks = tifffile.imread(mask_fp)

print("raw shape:", raw.shape)
print("mask shape:", cp_masks.shape, "labels:", int(cp_masks.max()))

raw shape: (512, 512)
mask shape: (512, 512) labels: 21


In [6]:
from pathlib import Path
import tifffile

RAW_DIR = Path("/QRISdata/Q6080/Q6080/IndividualLabFolders/AG/DRGAnushka/resonant")

candidates = sorted(
    [f for f in RAW_DIR.glob("*.tif") if not f.name.startswith(".")] +
    [f for f in RAW_DIR.glob("*.tiff") if not f.name.startswith(".")]
)

good = []
bad = []
for f in candidates:
    try:
        with tifffile.TiffFile(f) as tif:
            _ = tif.pages[0].shape   # touch first page only
        good.append(f)
    except Exception as e:
        bad.append((f.name, str(e)))

print("Candidates:", len(candidates))
print("Good:", len(good))
print("Bad:", len(bad))
if bad:
    print("Example bad:", bad[0])

Candidates: 22
Good: 22
Bad: 0


In [7]:
raw_fp = good[0]
print("Using:", raw_fp)

with tifffile.TiffFile(raw_fp) as tif:
    print("Num pages:", len(tif.pages))
    print("Page[0] shape:", tif.pages[0].shape, "dtype:", tif.pages[0].dtype)
    frame0 = tif.pages[0].asarray()   # loads only first frame

print("Loaded single frame:", frame0.shape, frame0.dtype)

Using: /QRISdata/Q6080/Q6080/IndividualLabFolders/AG/DRGAnushka/resonant/614399S2D20250113.tif
Num pages: 1
Page[0] shape: (512, 512) dtype: uint16
Loaded single frame: (512, 512) uint16


In [8]:
MASK_DIR = Path.home() / "cellpose_masks_resonant"
mask_fp = MASK_DIR / f"{raw_fp.stem}_masks.tif"

cp_masks = tifffile.imread(mask_fp)
print("Mask:", cp_masks.shape, cp_masks.dtype, "labels:", int(cp_masks.max()))

Mask: (512, 512) uint16 labels: 21


In [9]:
import numpy as np

im2d = frame0.astype(np.float32)
im2d = (im2d - im2d.min()) / (im2d.max() - im2d.min() + 1e-8)

rgb = np.repeat((im2d*255).astype(np.uint8)[...,None], 3, axis=-1)
http://localhost:8888/notebooks/cellposeSAM/Untitled1.ipynb?kernel_name=python3#
predictor.set_image(rgb)
print("SAM embedding done")

SyntaxError: invalid syntax (600296233.py, line 7)